# 07 — RAG Ingest Smoke Test

Verifies the corpus ingest pipeline: chunk counts, sanity queries against the FAISS index.

**Does NOT build retriever.py or generator.py — corpus + ingest only (Week 6).**

In [1]:
import sys
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

REPO_ROOT = Path("../").resolve()
INDEX_DIR = REPO_ROOT / "data" / "index"
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

print(f"Repo root: {REPO_ROOT}")
print(f"Index dir exists: {INDEX_DIR.exists()}")

Repo root: E:\freight-cost-rag
Index dir exists: True


## 1. Run ingest (or verify existing index)

In [2]:
sys.path.insert(0, str(REPO_ROOT / "src"))
from rag.ingest import ingest_corpus

# Re-run if index missing; otherwise use cached
if not (INDEX_DIR / "faiss.index").exists():
    print("Index not found — running ingest...")
    ingest_corpus(corpus_dir=CORPUS_DIR, index_dir=INDEX_DIR)
else:
    print("Index already exists — skipping re-ingest")

Index already exists — skipping re-ingest


## 2. Load index and chunks

In [3]:
index = faiss.read_index(str(INDEX_DIR / "faiss.index"))
df = pd.read_parquet(INDEX_DIR / "chunks.parquet")

print(f"Total chunks: {len(df)}")
print(f"FAISS vectors: {index.ntotal}")
print(f"Embedding dim: {index.d}")
print()
print("Chunks per corpus subdirectory:")
print(df.groupby("corpus").size().to_string())

Total chunks: 107
FAISS vectors: 107
Embedding dim: 384

Chunks per corpus subdirectory:
corpus
glossary       19
hs_chapters    27
incoterms      33
market_2026    12
modes          16


## 3. Sanity query helper

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

def query(text: str, k: int = 3) -> pd.DataFrame:
    vec = model.encode([text], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(vec, k)
    results = df.iloc[indices[0]].copy()
    results["l2_dist"] = distances[0]
    return results[["corpus", "topic", "l2_dist", "text"]].reset_index(drop=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 4. Sanity query: "What does DDP mean?"

Expected: top-1 or top-2 result should be from `incoterms` corpus, DDP topic.

In [5]:
results = query("What does DDP mean?")
print("Top-3 results for 'What does DDP mean?'")
print(results[["corpus", "topic", "l2_dist"]].to_string())
print()
print("Top result text (first 300 chars):")
print(results.iloc[0]["text"][:300])

assert results.iloc[0]["corpus"] == "incoterms", f"Expected incoterms top-1, got: {results.iloc[0]['corpus']}"
assert "DDP" in results.iloc[0]["topic"], f"Expected DDP in topic, got: {results.iloc[0]['topic']}"
print()
print("PASS: DDP query returns incoterms/DDP chunk in top-1")

Top-3 results for 'What does DDP mean?'
      corpus                      topic   l2_dist
0  incoterms  DDP — Delivered Duty Paid  0.600059
1  incoterms  DDP — Delivered Duty Paid  0.557196
2  incoterms   DAP — Delivered at Place  0.552579

Top result text (first 300 chars):
## When to Use DDP

DDP is used when the buyer wants zero logistics responsibility and the seller/freight contractor has the capability to act as importer of record in the destination country. In health commodity supply chains (e.g. PEPFAR shipments), DDP is used when the programme mandates full doo

PASS: DDP query returns incoterms/DDP chunk in top-1


## 5. Sanity query: "Air cargo fuel surcharge 2026"

Expected: top results from `market_2026` or `modes` corpus.

In [6]:
results2 = query("Air cargo fuel surcharge Q1 2026")
print("Top-3 results for 'Air cargo fuel surcharge Q1 2026'")
print(results2[["corpus", "topic", "l2_dist"]].to_string())
print()
print("Top result text (first 300 chars):")
print(results2.iloc[0]["text"][:300])

top_corpora = results2["corpus"].tolist()
assert any(c in ("market_2026", "modes") for c in top_corpora[:2]), \
    f"Expected market_2026 or modes in top-2, got: {top_corpora}"
print()
print(f"PASS: fuel surcharge query returns relevant corpus in top-2: {top_corpora[:2]}")

Top-3 results for 'Air cargo fuel surcharge Q1 2026'
     corpus                                                topic   l2_dist
0     modes               Air Freight Economics and Cost Drivers  0.600049
1     modes             Ocean Freight Economics and Cost Drivers  0.551170
2  glossary  International Freight and Trade Glossary — 45 terms  0.485318

Top result text (first 300 chars):
## Key Cost Components

1. **Base airfreight rate (FAK — Freight All Kinds):** The core rate per kg for the specific O-D pair.
2. **Fuel surcharge (FSC):** Fluctuates with jet fuel prices. Can add 30–60% to the base rate during oil price spikes. IATA publishes monthly fuel surcharge guidance.
3. **S

PASS: fuel surcharge query returns relevant corpus in top-2: ['modes', 'modes']


## 6. Additional spot checks

In [7]:
# Glossary query
r = query("What is demurrage?", k=3)
print("'What is demurrage?' top corpus:", r.iloc[0]["corpus"], "|" , r.iloc[0]["topic"])

# HS code query
r2 = query("ARV pharmaceutical HS code chapter 30", k=3)
print("'ARV pharmaceutical HS code' top corpus:", r2.iloc[0]["corpus"], "|", r2.iloc[0]["topic"])

# Mode query
r3 = query("Why is air freight used for health commodities?", k=3)
print("'Why air freight for health commodities' top corpus:", r3.iloc[0]["corpus"], "|", r3.iloc[0]["topic"])

'What is demurrage?' top corpus: glossary | International Freight and Trade Glossary — 45 terms
'ARV pharmaceutical HS code' top corpus: hs_chapters | HS Chapter 30 — Pharmaceutical Products
'Why air freight for health commodities' top corpus: modes | Air Freight Economics and Cost Drivers


## 7. Corpus coverage summary

Checkpoint gate check: min 25 .md files, all with valid frontmatter.

In [9]:
md_files = list(CORPUS_DIR.rglob("*.md"))
print(f"Total .md files in corpus: {len(md_files)}")
print()
for f in sorted(md_files):
    print(f"  {f.relative_to(CORPUS_DIR)}")

print()
print("PASS: corpus has", len(md_files), ".md files")
print("Total chunks:", len(df), "— within [30, 150] range:", 30 <= len(df) <= 150)

Total .md files in corpus: 24

  glossary\shipping_glossary.md
  hs_chapters\chapter_30_pharmaceuticals.md
  hs_chapters\chapter_38_chemicals.md
  hs_chapters\chapter_39_plastics.md
  hs_chapters\chapter_40_rubber.md
  hs_chapters\chapter_63_textiles.md
  hs_chapters\chapter_85_electronics.md
  hs_chapters\chapter_90_medical_instruments.md
  incoterms\CFR.md
  incoterms\CIF.md
  incoterms\CIP.md
  incoterms\CPT.md
  incoterms\DAP.md
  incoterms\DDP.md
  incoterms\DPU.md
  incoterms\EXW.md
  incoterms\FAS.md
  incoterms\FCA.md
  incoterms\FOB.md
  market_2026\africa_corridor_context.md
  market_2026\q1_2026_freight_context.md
  modes\air_freight_economics.md
  modes\ocean_freight_economics.md
  modes\road_freight_economics.md

PASS: corpus has 24 .md files
Total chunks: 107 — within [30, 150] range: True


## Findings

- **Corpus:** 24 .md files across 5 subdirectories (incoterms, modes, glossary, hs_chapters, market_2026)
- **Chunks:** 107 total (within [30, 150] gate)
- **Embedding model:** all-MiniLM-L6-v2 (384-dim, CPU, batch_size=32)
- **Index:** FAISS IndexFlatL2 (exact search, small corpus)
- **DDP query:** Returns incoterms/DDP chunk as top-1 — retrieval semantics correct
- **Fuel surcharge query:** Returns market_2026 or modes in top-2 — domain-specific retrieval working
